In [ ]:
import pandas as pd
import json

In [ ]:
df=pd.read_parquet('Dataset_Master_Predictions_2026.parquet')
df.head()

In [ ]:
df.shape

#### Vérification des SIREN pour sample

In [ ]:
# 1. Préparation des données (Assurer le format String pour éviter les pertes de 0)
df['SIREN'] = df['SIREN'].astype(str).str.strip().str.zfill(9)
col_dep = "Code du département de l'établissement"

# 2. Filtrage des SIREN commençant par '0'
df_zero = df[df['SIREN'].str.startswith('0')].copy()
nb_zero = len(df_zero)

print(f"🔍 ANALYSE DES {nb_zero} SIREN COMMENÇANT PAR ZÉRO")
print("-" * 50)

# --- VÉRIFICATION 1 : INTÉGRITÉ ET LONGUEUR ---
anomalies_longueur = df_zero[df_zero['SIREN'].apply(len) != 9]
if anomalies_longueur.empty:
    print(f"✅ Longueur : Tous les SIREN '0' font bien 9 caractères.")
else:
    print(f"⚠️ Anomalie : {len(anomalies_longueur)} SIREN n'ont pas la bonne taille.")

# --- VÉRIFICATION 2 : UNICITÉ (DOUBLONS) ---
doublons = df_zero.duplicated(subset=['SIREN']).sum()
if doublons == 0:
    print(f"✅ Unicité : Aucun doublon détecté parmi ces SIREN.")
else:
    print(f"⚠️ Attention : {doublons} SIREN apparaissent plusieurs fois.")

# --- VÉRIFICATION 3 : RÉPARTITION GÉOGRAPHIQUE ---
print(f"\n📍 RÉPARTITION PAR DÉPARTEMENT (Top 10) :")
repartition = df_zero[col_dep].value_counts().head(10)
print(repartition)

# --- VÉRIFICATION 4 : ÉCHANTILLON VISUEL ---
print(f"\n👀 APERÇU DES PREMIÈRES LIGNES :")
cols_view = ['SIREN', 'Dénomination', col_dep, 'Prob_1an']
print(df_zero[cols_view].head())

#### Lancement de l'extraction du sample sur 500 SIREN pour récupération des bilans

In [ ]:
# # 1. Nettoyage et formatage
# df_clean = df.drop_duplicates(subset=['SIREN']).copy()
# df_clean['SIREN'] = df_clean['SIREN'].astype(str).str.zfill(9)

# # 2. Sélection des profils prioritaires (Risque N+1 et N+3)
# top_risques = pd.concat([
#     df_clean.nlargest(200, 'Prob_1an'),
#     df_clean.nlargest(200, 'Prob_3ans')
# ]).drop_duplicates(subset=['SIREN'])

# # 3. Compléter pour atteindre exactement 500
# nb_actuel = len(top_risques)
# nb_a_ajouter = 500 - nb_actuel

# if nb_a_ajouter > 0:
#     reste_df = df_clean[~df_clean['SIREN'].isin(top_risques['SIREN'])]
#     complement = reste_df.sample(n=min(nb_a_ajouter, len(reste_df)))
#     sample_final = pd.concat([top_risques, complement])
# else:
#     sample_final = top_risques.head(500)

# # 4. Export en format LISTE SIMPLE (format JSON array)

# siren_list = sample_final['SIREN'].tolist()

# with open('siren_sample.json', 'w', encoding='utf-8') as f:
#     json.dump(siren_list, f, indent=2)

# print(f"✅ Fichier 'siren_sample.json' généré avec {len(siren_list)} SIREN.")
# print(f"   Structure : ['{siren_list[0]}', '{siren_list[1]}', ...]")

#### Augmentation de l'échantillon à 2000 SIREN

In [ ]:
# --- CONFIGURATION ---
TAILLE_CIBLE = 2000
NB_RISQUE_TOP = 400 

# 1. Nettoyage
df_clean = df.drop_duplicates(subset=['SIREN']).copy()
df_clean['SIREN'] = df_clean['SIREN'].astype(str).str.zfill(9)

# 2. Sélection (Risque N+1 et N+3)
top_risques = pd.concat([
    df_clean.nlargest(NB_RISQUE_TOP, 'Prob_1an'),
    df_clean.nlargest(NB_RISQUE_TOP, 'Prob_3ans')
]).drop_duplicates(subset=['SIREN'])

# 3. Compléter jusqu'à 2000
nb_a_ajouter = TAILLE_CIBLE - len(top_risques)
reste_df = df_clean[~df_clean['SIREN'].isin(top_risques['SIREN'])]
complement = reste_df.sample(n=min(nb_a_ajouter, len(reste_df)))
sample_final = pd.concat([top_risques, complement])

# 4. Export
siren_list = sample_final['SIREN'].tolist()
with open('siren_batch_2000.json', 'w', encoding='utf-8') as f:
    json.dump(siren_list, f, indent=2)

print(f"✅ Nouveau fichier 'siren_batch_2000.json' prêt !")